<a href="https://colab.research.google.com/github/AliceFranca0/tcc-deteccao-fake-news-ptbr/blob/main/notebooks/Fase6_FakeNews_Sinteticas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/TCC"

Mounted at /content/drive


In [2]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 17.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [3]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [4]:
from google.colab import userdata
userdata.get('GEMINI_API_KEY')

'AQ.Ab8RN6L17-eC0mp1q1MRR0GWyYOhSeaAPTDC9UQlwzrqEeMBFg'

In [5]:
import pandas as pd, time

MODELO_LLM = "gemini-3.5-flash-lite"
CAMINHO_SAIDA = f"{BASE}/dados/fake_news_sinteticas_gemini35.csv"

df = pd.read_csv(f"{BASE}/dados/fakerecogna_bruto.csv")
titulos = df[df["Classe"] == 1]["Titulo"].dropna().sample(150, random_state=42).tolist()

PROMPT = """Você está auxiliando uma pesquisa acadêmica sobre detecção automática
de desinformação. Reescreva a manchete abaixo como se fosse uma notícia falsa
sensacionalista, em português brasileiro, com 60 a 120 palavras.
Este texto será usado APENAS como amostra rotulada para testar classificadores.

Manchete original: {titulo}

Responda somente com o texto da notícia, sem comentários."""

resultados = []
falhas = []

for i, titulo in enumerate(titulos):
    sucesso = False

    # Até 3 tentativas por item, com pausa crescente
    for tentativa in range(3):
        try:
            r = client.models.generate_content(
                model=MODELO_LLM,
                contents=PROMPT.format(titulo=titulo),
            )
            resultados.append({"titulo_origem": titulo, "texto_sintetico": r.text.strip()})
            sucesso = True
            break
        except Exception as e:
            if tentativa < 2:
                time.sleep(10 * (tentativa + 1))   # 10s, depois 20s
            else:
                print(f"Falhou no item {i} após 3 tentativas: {str(e)[:100]}")
                falhas.append({"indice": i, "titulo": titulo, "erro": str(e)[:200]})

    if i == 0 and not sucesso:
        print("O primeiro item falhou — verifique o modelo antes de continuar.")
        break

    time.sleep(7)

    if (i + 1) % 10 == 0:
        pd.DataFrame(resultados).to_csv(CAMINHO_SAIDA, index=False, encoding="utf-8-sig")
        print(f"{i+1}/{len(titulos)} processados — {len(resultados)} gerados, {len(falhas)} falhas")

pd.DataFrame(resultados).to_csv(CAMINHO_SAIDA, index=False, encoding="utf-8-sig")
print(f"\nTotal gerado: {len(resultados)} | Falhas: {len(falhas)}")

if falhas:
    print("\nDetalhe das falhas:")
    for f in falhas:
        print(f"  item {f['indice']}: {f['erro'][:120]}")

10/150 processados — 10 gerados, 0 falhas
20/150 processados — 20 gerados, 0 falhas
30/150 processados — 30 gerados, 0 falhas
40/150 processados — 40 gerados, 0 falhas
50/150 processados — 50 gerados, 0 falhas
60/150 processados — 60 gerados, 0 falhas
70/150 processados — 70 gerados, 0 falhas
80/150 processados — 80 gerados, 0 falhas
90/150 processados — 90 gerados, 0 falhas
100/150 processados — 100 gerados, 0 falhas
110/150 processados — 110 gerados, 0 falhas
120/150 processados — 120 gerados, 0 falhas
130/150 processados — 130 gerados, 0 falhas
140/150 processados — 140 gerados, 0 falhas
150/150 processados — 150 gerados, 0 falhas

Total gerado: 150 | Falhas: 0


In [6]:
import pandas as pd

sint = pd.read_csv(f"{BASE}/dados/fake_news_sinteticas_gemini35.csv")

print(f"{len(sint)} notícias sintéticas")
print(f"Tamanho médio: {sint['texto_sintetico'].str.split().str.len().mean():.1f} palavras")
print(f"Mediana: {sint['texto_sintetico'].str.split().str.len().median():.0f} palavras")
print("\nExemplo:\n")
print(sint["texto_sintetico"].iloc[0][:400])

150 notícias sintéticas
Tamanho médio: 108.8 palavras
Mediana: 109 palavras

Exemplo:

URGENTE! GOVERNO ESPANHOL ESCONDE A VERDADE: Vírus mutante escapa de laboratório secreto enquanto ditadores comunistas decretam o fim falso da quarentena para destruir a economia! Documentos vazados por um hacker anônimo revelam que o plano satânico de desconfinamento em quatro passos foi ordenado por potências estrangeiras para dizimar a população idosa e implantar chips de controle mental atravé


In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
CAMINHO_MODELO = f"{BASE}/modelos/modelo_bertimbau_final"

tok = AutoTokenizer.from_pretrained(CAMINHO_MODELO)
mod = AutoModelForSequenceClassification.from_pretrained(CAMINHO_MODELO).to(dispositivo).eval()

print("Modelo carregado em:", dispositivo)

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Modelo carregado em: cpu


In [10]:
import pandas as pd

sint = pd.read_csv(f"{BASE}/dados/fake_news_sinteticas.csv")
textos = sint["texto_sintetico"].tolist()

import joblib
vetorizador = joblib.load(f"{BASE}/modelos/vetorizador_tfidf.pkl")
modelo_lr = joblib.load(f"{BASE}/modelos/modelo_logistic_regression.pkl")
modelo_rf = joblib.load(f"{BASE}/modelos/modelo_random_forest.pkl")

X_sint = vetorizador.transform(textos)
det_lr = (modelo_lr.predict(X_sint) == 0).mean()
det_rf = (modelo_rf.predict(X_sint) == 0).mean()

# BERTimbau
entradas = tok(textos, truncation=True, padding=True, max_length=256, return_tensors="pt").to(dispositivo)
with torch.no_grad():
    pred_bert = mod(**entradas).logits.argmax(-1).cpu().numpy()
det_bert = (pred_bert == 0).mean()

# Acurácias no corpus original — substitua pelos SEUS valores
deteccao = pd.DataFrame({
    "modelo": ["Logistic Regression", "Random Forest", "BERTimbau"],
    "acuracia_original": [0.944, 0.937, 0.000],   # <- ajuste o do BERTimbau
    "deteccao_sintetica": [det_lr, det_rf, det_bert],
})

deteccao.to_csv(f"{BASE}/dados/deteccao_sinteticas.csv", index=False)
print(deteccao.round(4))

                modelo  acuracia_original  deteccao_sintetica
0  Logistic Regression              0.944              0.6463
1        Random Forest              0.937              0.4830
2            BERTimbau              0.000              0.2041
